# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR⁲ dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All entities are referenced by their Croissant `@id` fields for clarity and interoperability.

### Dataset Source
The dataset is described via a Croissant schema JSON-LD file.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Display dataset title and description
print("Dataset Title: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells")
print("This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.")


## 2. Data Overview
Review available record sets, fields, and their Croissant `@id` fields.

The Croissant standard organizes dataset content as *record sets*, each defining one or more *fields*, which may correspond to columns in tabular data. Let's enumerate the available record sets, their fields, and field data types using the `mlcroissant` library.

In [ ]:
# Get all record set @ids and some core descriptions
record_sets = dataset.metadata.record_sets
print(f"Number of record sets found: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record set name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field name: {field.name} | @id: {field.id} | Data type: {field.data_type}")
    print()
# We'll use the first record set as the main example below:
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame. All references are made by Croissant `@id`.

We'll load all record sets and inspect the first few rows for one of them.

In [ ]:
# Load all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set {record_set_id}")

# Check columns in the main record set, if found
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Fields (@id) in main record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set found or it contains no data.")

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field for filtering, normalization, and grouping, using only Croissant `@id` fields.

In [ ]:
# Example selection: find a numeric field (@id) to analyze
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Try finding a float/int column
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the first numeric field found (referenced by @id)
        print(f"Using numeric field (@id): {numeric_field}\n")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > average ({threshold:.3f}): {len(filtered_df)} records")

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print("Normalized field example:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

        # Try grouping by a categorical (string/object) field
        group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            print(f"\nGrouping by field (@id): {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main record set with data for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if applicable, its values grouped by the chosen categorical field. All references are by Croissant `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        # Plot mean values by group
        group_means = dataframes[main_record_set_id].groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:10]
        plt.figure(figsize=(10,6))
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.title(f'Mean {numeric_field} by {group_field} (top 10 groups)')
        plt.xlabel(f'Mean {numeric_field}')
        plt.ylabel(group_field)
        plt.show()
else:
    print("Visualization unavailable: no data loaded or no numeric field selected.")

## 6. Conclusion

- The FAIR⁲ dataset provides detailed protein-level data from mass spectrometry experiments, richly described and organized via the Croissant schema with interoperable `@id` references.
- Using `mlcroissant`, we efficiently listed, loaded, and explored the dataset's record sets and fields.
- Exploratory analysis (by field `@id`) showed how to filter and normalize values, as well as group and visualize results.
- This notebook provides a reproducible, standards-based workflow for initial FAIR⁲ data analysis and can be extended for more advanced studies such as biomarker identification, outlier analysis, or downstream machine learning tasks.
